# 01 · Data: census, snapshots, splits
1. **Census** every dataset *before* mapping: raw labels, surface match rate, offset issues. Any `unmapped` label must be added to `s1pii/taxonomy.py` in a reviewed commit (bump `MAP_VERSION`).
2. **Materialize** calibration/test splits as canonical JSONL snapshots (recorded revisions and hashes).
3. Sync snapshots to Drive.

In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')
if not REPO.exists():
    from google.colab import userdata                    # add GH_TOKEN (fine-grained, read-only) in Colab secrets
    tok = userdata.get('GH_TOKEN')
    subprocess.run(['git', 'clone', f'https://{tok}@github.com/anitksahu/s1-pii.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', 'https://github.com/anitksahu/s1-pii.git'], check=True)
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -e '.[data]'

In [ ]:
import json
from s1pii.data.census import census
HEADLINE = ['tab_direct', 'spy_medical', 'spy_legal', 'pii_trace', 'nemotron']
SECONDARY = ['tab_quasi', 'gretel']
for n in HEADLINE + SECONDARY:
    r = census(n)
    print(n, r['status'], 'unmapped=', r.get('unmapped'), 'surface_match=', r.get('surface_match_rate'), 'issues=', r.get('issues'), r.get('error', ''))

In [ ]:
from s1pii import bench
for n in HEADLINE + SECONDARY:
    print(n, bench.materialize(n))

In [ ]:
from s1pii.data.loaders import snapshot_meta
from s1pii.schema import read_jsonl
for n in HEADLINE + SECONDARY:
    p = bench.split_paths(n)
    print(n, {k: len(read_jsonl(v)) for k, v in p.items()})
# HF revisions resolved at first load: copy them into s1pii/data/manifest.yaml (revision: <sha>) and commit
import glob
for m in sorted(glob.glob(os.environ['S1PII_DATA'] + '/snapshots/*.meta.json')):
    j = json.load(open(m)); print(j['name'], j['split'], j.get('hf_revision') or j.get('source_sha256'), j['dataset_hash'], j['report']['rejected'])

In [ ]:
!rsync -a $S1PII_DATA/ /content/drive/MyDrive/s1pii/data/